# freMTPL claim frequency with the pricing pipeline

This executable tutorial follows the library's ingestion, training, and publication APIs on a reproducible 10,000-policy sample of `freMTPL2freq`. It creates a verified model frame, cross-validation metrics, a rating workbook, and persistent SQLite audit records. No SQL Server or credentials are needed.

Run `uv sync --locked --extra notebook` from the repository root, select `.venv/bin/python` as your notebook kernel, and run the cells in order. The first run downloads the public data through the library's OpenML loader; later runs use scikit-learn's download cache. The row limit reduces fitting time, not download size.

Save this notebook before fitting. The pipeline records its saved source as model evidence. Outputs and execution counts are excluded from that identity.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import text
from superglm import Categorical, Spline, SuperGLM

from pricing_pipeline.data.fremtpl import fetch_fremtpl, prepare_fremtpl_raw_frame
from pricing_pipeline.models.config import ValidationSplitConfig
from pricing_pipeline.notebook import (
    PricingModelSpec,
    fit_model,
    connect,
    load_model_frame,
    save_model_version,
    register_model,
    save_model_frame,
)

REPO_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "tutorials/fremtpl_frequency/demo.ipynb").is_file()
)
SOURCE_ROOT = REPO_ROOT / "tutorials/fremtpl_frequency"
ROWS = 10_000  # Set to None to fit all eligible policies.
SEED = 42
DATA_AS_OF = "2026-09-01"  # Illustrative snapshot label, explained below.
LOCAL_ROOT = REPO_ROOT / "state/fremtpl_frequency" / f"demo-ppform-{ROWS}-seed-{SEED}"
FRAME_PATH = LOCAL_ROOT / "model_frame.joblib"


## 1. Load and prepare the model frame

`ClaimNb` is the claim count and `Exposure` is the insured duration in years. This example drops rows with nonpositive or nonfinite exposure, preserves positive exposure and claim counts without capping them, and adds `log(Exposure)` as the Poisson offset. `LogDensity` is a deterministic feature transformation. Vehicle age is capped at 20 years, as in the upstream freMTPL walkthrough. This pools the sparse older-vehicle tail, which includes recorded ages of 99 and 100 years, and avoids an unsupported spline fit in this small sample. The cap is a demo modelling decision to revisit for real work.

The equation is `expected claims = Exposure * exp(model effects)`. Exposure is not also used as a fitting weight. `rating_weight` weights exported rating summaries by exposure and does not change the fitted likelihood.

The public data does not provide a reporting cutoff for this example. `DATA_AS_OF` is an explicitly illustrative snapshot label, not a claim about source completeness. For your dataset, replace it with the actual data-as-at date. A random validation split here measures performance on similar policies, not future-period performance.


In [ ]:
raw = prepare_fremtpl_raw_frame(fetch_fremtpl())
exposure = raw["Exposure"].astype(float)
frame = raw.loc[np.isfinite(exposure) & exposure.gt(0)].copy()
if ROWS is not None:
    if ROWS < 1_000:
        raise ValueError("Use at least 1,000 policies for this demo.")
    frame = frame.sample(n=min(ROWS, len(frame)), random_state=SEED)
frame = frame.sort_values("IDpol").reset_index(drop=True)
frame["Exposure"] = frame["Exposure"].astype(float)
frame["log_exposure"] = np.log(frame["Exposure"])
frame["VehAge"] = frame["VehAge"].clip(upper=20)
frame["LogDensity"] = np.log1p(frame["Density"].astype(float))
frame["rating_weight"] = frame["Exposure"]
frame["data_as_of"] = DATA_AS_OF
for column in ("Area", "VehGas"):
    frame[column] = frame[column].astype(str)

FEATURES = ("DrivAge", "VehAge", "BonusMalus", "LogDensity", "Area", "VehGas")
frame = frame.loc[:, [
    "IDpol", "ClaimNb", "Exposure", "log_exposure", "rating_weight",
    "data_as_of", *FEATURES,
]]
assert frame["IDpol"].is_unique
assert frame.notna().all().all()
display(pd.Series({
    "policies": len(frame),
    "claims": frame["ClaimNb"].sum(),
    "exposure_years": frame["Exposure"].sum(),
    "observed_annual_frequency": frame["ClaimNb"].sum() / frame["Exposure"].sum(),
}))
display(frame.head())


## 2. Save the handoff and connect locally

`save_model_frame` writes the frame and a hash receipt; `load_model_frame` verifies both before loading. An identical rerun reuses the handoff. If you change preprocessing or the snapshot date, choose a new `LOCAL_ROOT` to retain the previous evidence.

`connect(mode="local")` creates persistent SQLite audit databases and artifact directories below `LOCAL_ROOT`.


In [ ]:
frame_artifact = save_model_frame(frame, FRAME_PATH)
frame = load_model_frame(FRAME_PATH)
pricing = connect(mode="local", local_root=LOCAL_ROOT)
display(pricing.destination)
display(frame_artifact)


## 3. Declare the model and validation

`PricingModelSpec` tells the pipeline which columns are features, keys, target, offset, and export weights. Two shuffled folds keep this tutorial small. `fit_reml` estimates spline smoothing penalties. The pipeline validates on held-out folds, then fits the final model on the whole sample.


In [ ]:
MODEL = PricingModelSpec(
    name="FREMTPL_DEMO_FREQUENCY",
    label="freMTPL demo claim frequency",
    target="ClaimNb",
    model_type="superglm_poisson",
    deployment_slot="FREMTPL_DEMO_UAT",
    features=FEATURES,
    dataset_name="fremtpl_demo_frequency_frame",
    source_system="OpenML freMTPL2freq, data_id=41214",
    pk_columns=("IDpol",),
    offset_column="log_exposure",
    offset_source_column="Exposure",
    offset_label="log(Exposure)",
    export_weight_column="rating_weight",
    data_as_of_column="data_as_of",
    validation=ValidationSplitConfig.kfold(n_splits=2, random_state=SEED),
    fit_mode="fit_reml",
)
model = register_model(pricing, MODEL, source_root=SOURCE_ROOT)
superglm_model = SuperGLM(
    family="poisson",
    selection_penalty=0.0,
    retain_fit_state=False,  # Drop training caches; keep prediction and summary.
    discrete=True,
    n_bins=64,
    features={
        "DrivAge": Spline(k=6),
        "VehAge": Spline(k=5),
        "BonusMalus": Spline(k=5),
        "LogDensity": Spline(k=5),
        "Area": Categorical(),
        "VehGas": Categorical(),
    },
)


## 4. Fit and review validation metrics

`fit_model` clones the estimator, performs validation and the final fit, and derives the manifest, split evidence, candidate artifact, and rating export. The original `superglm_model` remains unfitted. The `cv_` metrics below come from held-out validation predictions; use these when judging predictive performance.


In [ ]:
candidate = fit_model(
    pricing,
    model=model,
    frame=frame,
    superglm_model=superglm_model,
    model_kind="RAW",
)
display(pd.Series(candidate.metrics, name="value").to_frame())


## Optional: save this model recipe

This writes the exact declared configuration used for the fit, including groups
and specials when configured. Reload with ModelRecipe.load(path).build(dataset=dataset)
for ordinary refitting. SQL assigns revisions when saving; deployment stays separate.


In [ ]:
# candidate.recipe.save(Path("state/fremtpl_model.toml"))


## 5. Publish the local candidate

Publication records a completed `RAW` package and its evidence in SQLite with status `LOCAL_AUDIT`. The query below inspects these local records. `list_model_versions` is for packages with remote `PUBLISHED` status and does not list local audit packages. Equivalent successful rating models on the same manifest are deduplicated, so an equivalent rerun can return the existing package version. Always use the returned publication paths, which also work after deduplication.

Local publication is an audit demonstration. Editor operations, manual-edit publication, and deployment require the guarded remote workflow described in the notebook guide.


In [ ]:
published = save_model_version(pricing, candidate)
display({
    "model": published.model_name,
    "kind": published.model_kind,
    "package_version": published.package_version,
    "recipe_revision": published.recipe_revision,
    "recipe_status": published.recipe_status,
    "package_status": published.package_status,
    "manifest_id": published.manifest_id,
    "reused_equivalent_package": published.deduplicated,
    "rating_workbook": published.rating_workbook_path,
})
with pricing.engine.connect() as connection:
    audit = pd.read_sql_query(
        text("""
            SELECT r.model_run_id, r.model_kind, r.manifest_id, r.run_status,
                   p.package_version, p.package_status
            FROM pricing.MODEL_RUN AS r
            JOIN pricing.PRICING_RATE_PACKAGE AS p
              ON p.rate_package_id = r.rate_package_id
            WHERE r.model_id = :model_id
            ORDER BY p.package_version DESC
        """),
        connection,
        params={"model_id": model.model_id},
    )
display(audit)


## 6. Inspect the exported curves

Spline main effects are exported as exact polynomial segments. In the workbook, `a`, `b`, `c`, and `d` define the log relativity inside each interval. The displayed `Relativity` is the value at the interval origin. It is not constant across that interval.

`V_FINAL_MODEL_RELATIVITY` exposes every effect with model and dataset dates for Power BI. Read `representation` to distinguish lookup entries from spline coefficients and per-unit factors. Evaluate a shared feature grid to compare curves over time. The SQL Server prediction procedure evaluates the polynomial at each policy's input value.

The exposure offset is represented by an exported factor. Annual frequency corresponds to one year of exposure; expected policy claims also include the policy's exposure factor.

These are fitted model outputs. The public-data demo has no business review or temporal validation.


In [ ]:
workbook_path = Path(published.rating_workbook_path)
assert workbook_path.is_file()
with pd.ExcelFile(workbook_path) as workbook:
    display(pd.DataFrame({"sheet": workbook.sheet_names}))
    display(pd.read_excel(workbook, sheet_name=0).head(12))
print(f"Workbook: {workbook_path}")
print(f"Local data and audit artifacts: {LOCAL_ROOT}")
with pricing.engine.connect() as connection:
    effects = pd.read_sql_query(
        text("""
            SELECT term_name, representation, level_value, level_sort_order,
                   relativity, log_coefficient, lower_bound, upper_bound,
                   upper_inclusive, a, b, c, d, data_as_of_date
            FROM pricing.V_FINAL_MODEL_RELATIVITY
            WHERE rate_package_id = :package_id
            ORDER BY term_sequence_no, level_sort_order, level_value
        """),
        connection,
        params={"package_id": published.rate_package_id},
    )
display(effects)
pricing.engine.dispose()


## Continue with a model repository

This tutorial combines ingestion and training so you can run the library in one notebook. For a maintained model, use `pricing-pipeline init`, edit `pricing_scaffold.toml`, then run `pricing-pipeline scaffold`. Move data preparation into notebook 01 and the model declaration into notebook 03. The generated workflow separates exploration, editing, manual adjustments, and deployment.

See [the notebook API guide](../../docs/notebooks/README.md) and [the project README](../../README.md) for the full workflow.
